# Model Training - Villa Vente Marrakech (Optimisation XGBoost)
Ce notebook utilise un modèle **XGBoost** optimisé pour atteindre plus de 80% de performance sur les villas.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.compose import TransformedTargetRegressor

# Ajouter le chemin vers la racine pour importer les helpers
sys.path.append('../')
from model_training_helpers import split_data, model_pipeline, metric_model, get_features, print_metrics, save_model, preprocess_data

print("Modules chargés avec succès.")

## 1. Chargement et Fusion des données

In [ ]:
csv_path = '../data/cleaned_data/vente/villa_vente_cleaned.csv'
raw_csv_path = '../data/marrakech_immo_vente/villa_vente.csv'

if os.path.exists(csv_path) and os.path.exists(raw_csv_path):
    df = pd.read_csv(csv_path)
    raw_df = pd.read_csv(raw_csv_path)
    
    # Fusion avec la description pour extraire les mots-clés NLP
    merge_cols = ['prix_num', 'surface_num', 'chambres_num', 'salles_bain_num', 'quartier']
    raw_subset = raw_df[merge_cols + ['description', 'localisation']].drop_duplicates(subset=merge_cols)
    df = pd.merge(df, raw_subset, on=merge_cols, how='left')
    
    print(f"Données chargées : {df.shape}")
else:
    print(f"ERREUR : Fichiers non trouvés.")

## 2. Préparation (Feature Engineering & Outliers)

In [ ]:
# Appliquer le prétraitement spécifique aux villas (filtres de prix et surface adaptés)
df = preprocess_data(df, property_type='villa')
print(f"Shape après filtrage des outliers : {df.shape}")

# Split des données
X_train, X_test, y_train, y_test = split_data(df, target='prix_num')

# Identification des colonnes
num_features, cat_features = get_features(X_train)
print(f"Nombre de features numériques : {len(num_features)}")
print(f"Nombre de features catégorielles : {len(cat_features)}")

## 3. Entraînement du modèle XGBoost

In [ ]:
from model_training_helpers import tune_model
from sklearn.compose import TransformedTargetRegressor

print("Optimisation avancée des hyperparamètres XGBoost...")

# On augmente n_iter pour une recherche plus fine
param_dist = {
    'n_estimators': [2500, 3500, 4500],
    'max_depth': [7, 8, 9, 10],
    'learning_rate': [0.005, 0.01, 0.02],
    'subsample': [0.8, 0.85, 0.9],
    'colsample_bytree': [0.8, 0.85, 0.9],
    'min_child_weight': [2, 3, 4]
}

# Nous allons transformer la cible en log AVANT le tuning pour que le scoring soit cohérent
y_train_log = np.log1p(y_train)

best_xgb_pipeline = tune_model(
    X_train, y_train_log, 
    num_features, cat_features, 
    xgb.XGBRegressor, 
    param_dist, 
    n_iter=20, 
    cv=3
)

# Le modèle best_xgb_pipeline contient déjà le prétraitement
# On l'utilise tel quel, mais on doit inverser l'exp à la fin
class LogInverter:
    def __init__(self, model):
        self.model = model
    def predict(self, X):
        return np.expm1(self.model.predict(X))

# Alternativement, on recrée le pipeline TransformedTargetRegressor avec le modèle trouvé
best_xgb_model = best_xgb_pipeline.named_steps['model']

pipeline = TransformedTargetRegressor(
    regressor=model_pipeline(num_features, cat_features, best_xgb_model),
    func=np.log1p,
    inverse_func=np.expm1
)

print("Entraînement final...")
pipeline.fit(X_train, y_train)
print("Modèle optimisé avec succès.")

## 4. Évaluation

In [ ]:
y_pred = pipeline.predict(X_test)
metrics = metric_model(y_test, y_pred, model_name="XGBoost Villas")
print_metrics(metrics, model_name="XGBoost Villas")

## 5. Sauvegarde

In [ ]:
save_model(pipeline, 'villa_vente_final_model.joblib')
print("Modèle final sauvegardé.")